# CapFinch — Synthetic Omnichannel Data Generator

Builds a small, internally-consistent fake dataset (6 tables) to develop and test the
analytics / KPI pipeline **before** the e-commerce site launches.

CapFinch has run a physical store on **Square POS** for a while and is now launching its
first website, so transactions arrive from two sources. Both land in a **single `orders`
table** tagged with a `channel` flag, with channel-specific columns left null on the other
side — the same way Square's Orders API mixes in-store and Square Online sales.

**Two anchor keys**
- `transaction_id` — PK of `orders` (one row per completed sale → revenue, AOV)
- `customer_id` — PK of `customers` (one row per person → repeat rate, demographics).
  **Nullable on `orders`**: most walk-ins are anonymous, and some online buyers check out as guests.

**Relationship chain**
```
customers ─┬─ sessions ── events          (online only)
           └─ orders ── order_items ── products
```

**Realism rules baked in**
| Rule | Detail |
|---|---|
| Store hours | Open every day 10:00–18:00; no in-store order outside that window |
| In-store time-of-day | Slow morning, lunch bump, late-afternoon peak |
| In-store day-of-week | Saturday busiest, Mon/Tue slowest |
| Online time-of-day | 24/7 with a 19:00–22:00 peak and a 02:00–06:00 trough |
| Timeline | In-store history predates launch; online orders start on `LAUNCH_DATE` |
| Identity capture | ~68% of in-store orders anonymous; ~12% of online orders are guests |
| Payment mix | Cash/gift card only in store; PayPal only online |
| Basket size | Larger in store, mostly single-item online |

**KPI targets (funnel metrics are online-only by construction)**
| Metric | Target |
|---|---|
| Conversion rate (online orders / sessions) | ~2% |
| Cart abandonment (1 − completed/carts) | ~68% |
| Repeat purchase (customers with ≥2 attributable orders) | ~20% |

Everything is produced as a pandas `DataFrame` first (no CSV yet). Scale up later by raising
`N_CUSTOMERS` — the ratios above are preserved. A commented CSV-export cell is at the bottom.


In [50]:
# Dependencies live in .venv — install with:  .venv/bin/python -m pip install -r requirements.txt
# (marimo must be launched from that same venv, or it won't see these packages)

import random
from itertools import count

import numpy as np
import pandas as pd
from faker import Faker

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
fake = Faker("en_US")
Faker.seed(SEED)

pd.set_option("display.max_columns", None)
print("Libraries ready.")


Libraries ready.


## Config, timeline & KPI targets

`N_CUSTOMERS` is the single knob for dataset size. Repeat buyers get a 2nd order, so
*attributable* orders ≈ `N_CUSTOMERS × (1 + repeat_rate)`; anonymous walk-ins and guest
checkouts are then layered on top to hit the identity-capture rates. Session and cart counts
are derived from the **online** orders only, so conversion (~2%) and abandonment (~68%) hold
at any scale.


In [40]:
# ---- Scale (raise this to grow every table; KPI ratios are preserved) ----
N_CUSTOMERS = 15
N_PRODUCTS = 30  # catalog holds 30 SKUs across 6 categories; lower this to trim it

# ---- Timeline: in-store history predates the website ----
TODAY = pd.Timestamp("2026-08-31").normalize()
HISTORY_START = TODAY - pd.DateOffset(months=18)
LAUNCH_DATE = (TODAY - pd.DateOffset(months=3)).normalize()  # first possible online order

# ---- KPI targets ----
TARGET_CONVERSION = 0.02   # online orders / total sessions
TARGET_ABANDONMENT = 0.68  # 1 - completed_carts / carts_created
TARGET_REPEAT = 0.20       # share of customers with >= 2 orders

# ---- Channel mix & identity capture ----
P_ONLINE_ATTRIBUTABLE = 0.40  # of known-customer orders, share placed on the website
INSTORE_ANON_RATE = 0.68      # in-store orders with no customer_id (no receipt captured)
ONLINE_GUEST_RATE = 0.12      # online orders checked out as a guest

# ---- Store hours: open daily 10:00-18:00 ----
STORE_OPEN_HOUR, STORE_CLOSE_HOUR = 10, 18
INSTORE_HOURS = list(range(STORE_OPEN_HOUR, STORE_CLOSE_HOUR))
INSTORE_HOUR_W = [0.06, 0.09, 0.12, 0.13, 0.12, 0.15, 0.18, 0.15]  # lunch bump, late-day peak
INSTORE_DOW_W = [0.10, 0.10, 0.12, 0.13, 0.16, 0.23, 0.16]         # Mon..Sun, Saturday busiest

# ---- Online traffic shape: 24/7, evening peak, overnight trough ----
ONLINE_HOURS = list(range(24))
ONLINE_HOUR_W = [
    0.030, 0.018, 0.010, 0.007, 0.006, 0.008, 0.014, 0.022,  # 00-07
    0.030, 0.035, 0.038, 0.042, 0.055, 0.045, 0.040, 0.040,  # 08-15
    0.042, 0.048, 0.060, 0.085, 0.100, 0.095, 0.070, 0.050,  # 16-23
]
ONLINE_DOW_W = [0.15, 0.13, 0.13, 0.13, 0.14, 0.15, 0.17]

# ---- Categorical vocabularies ----
GENDERS = ["female", "male", "nonbinary"]
ACQ_SOURCES = ["organic", "mailchimp", "social", "referral"]
DEVICES = ["mobile", "desktop", "tablet"]
DEVICE_W = [0.62, 0.30, 0.08]
LANDING_PAGES = ["/", "/new-arrivals", "/sale", "/collections/best-sellers", "/product"]
EVENT_TYPES = ["page_view", "product_view", "add_to_cart", "checkout_start", "purchase"]

PAYMENT_METHODS = {
    "in_store": (["card_present", "cash", "apple_pay", "gift_card"], [0.60, 0.20, 0.15, 0.05]),
    "online": (["card_not_present", "paypal", "apple_pay"], [0.65, 0.20, 0.15]),
}
DECLINE_RATE = {"in_store": 0.02, "online": 0.05}  # card-present rarely fails
BASKET_SIZE = {
    "in_store": ([1, 2, 3, 4, 5], [0.30, 0.32, 0.20, 0.12, 0.06]),
    "online": ([1, 2, 3], [0.62, 0.28, 0.10]),
}

ENTRY_METHODS = ["chip", "tap", "swipe"]
ENTRY_METHOD_W = [0.50, 0.42, 0.08]
REGISTERS = ["REG01", "REG02"]
EMPLOYEES = ["EMP01", "EMP02", "EMP03", "EMP04"]
PROMO_CODES = ["WELCOME10", "SUMMER5", "MAILCHIMP15"]
FREE_SHIP_THRESHOLD = 75.0
SHIPPING_FEE = 6.95

repeat_customers = round(N_CUSTOMERS * TARGET_REPEAT)
print(f"Config: {N_CUSTOMERS} customers ({repeat_customers} repeat buyers), {N_PRODUCTS} products")
print(f"History {HISTORY_START.date()} -> {TODAY.date()}  |  site launched {LAUNCH_DATE.date()}")


Config: 15 customers (3 repeat buyers), 30 products
History 2025-02-28 -> 2026-08-31  |  site launched 2026-05-31


## Timestamp sampling

The single most visible tell of synthetic transaction data is uniformly-spread timestamps.
`sample_datetime` draws a day weighted by day-of-week, then an hour weighted by time-of-day,
using a different shape per channel. In-store draws are confined to store hours, and the
closing hour is tapered because nobody walks in at 17:55.


In [41]:
def _weighted_days(start, end, dow_w):
    days = pd.date_range(pd.Timestamp(start).normalize(), pd.Timestamp(end).normalize(), freq="D")
    w = np.array([dow_w[d.weekday()] for d in days], dtype=float)
    return days, w / w.sum()


def sample_datetime(channel, start, end):
    """Draw an order timestamp with channel-appropriate day-of-week and time-of-day shape."""
    dow_w = INSTORE_DOW_W if channel == "in_store" else ONLINE_DOW_W
    days, day_p = _weighted_days(start, end, dow_w)
    day = days[np.random.choice(len(days), p=day_p)]

    if channel == "in_store":
        hour_w = np.array(INSTORE_HOUR_W) / sum(INSTORE_HOUR_W)
        hour = int(np.random.choice(INSTORE_HOURS, p=hour_w))
        # taper the closing hour: no walk-ins in the last 15 minutes
        minute = int(np.random.randint(0, 45 if hour == STORE_CLOSE_HOUR - 1 else 60))
    else:
        hour_w = np.array(ONLINE_HOUR_W) / sum(ONLINE_HOUR_W)
        hour = int(np.random.choice(ONLINE_HOURS, p=hour_w))
        minute = int(np.random.randint(0, 60))

    return day + pd.Timedelta(hours=hour, minutes=minute, seconds=int(np.random.randint(0, 60)))


_demo = pd.Series([sample_datetime("in_store", HISTORY_START, TODAY).hour for _ in range(500)])
print("In-store hour range:", _demo.min(), "-", _demo.max(), "(store open 10-18)")


In-store hour range: 10 - 17 (store open 10-18)


## Table 4 — `products`  [PK: product_id]
Built first because orders/order_items and events reference it.

CapFinch is a gift-and-everyday boutique, so the catalog spans six categories mixing low-ticket
impulse/gift items with a few higher-ticket anchors:

| Category | Envelope | What's in it |
|---|---|---|
| Stationery | ~$5–40 | Cards, notebooks, pens, planners, washi tape |
| Home | ~$24–110 | Candles, vases, frames, diffusers, throws |
| Accessories | ~$18–88 | Jewelry, hats, scarves, small leather goods |
| Kitchen & Table | ~$12–95 | Mugs, tea towels, boards, salt cellars, kettles |
| Bath & Body | ~$8–56 | Soap, hand cream, bath salts, lip balm, body oil |
| Pantry & Treats | ~$6–26 | Honey, chocolate, tea, spiced nuts, jam |

Each product carries its own narrow price range rather than drawing from the whole category
envelope — a greeting card should never price out at $38. Prices are then snapped to retail-looking
endings (`.00` / `.50` / `.95`). Cost ratios vary by category: jewelry and bath carry the best
margin, pantry food the worst. Stock depth is inverse to price — cheap impulse items are stocked
deep, high-ticket anchors are stocked thin.


In [42]:
# (product_name, category, price_low, price_high)
CATALOG = [
    ("Letterpress Greeting Card", "Stationery", 5, 8),
    ("A5 Linen Notebook", "Stationery", 16, 24),
    ("Brass Fountain Pen", "Stationery", 28, 40),
    ("Weekly Desk Planner", "Stationery", 18, 28),
    ("Washi Tape Trio", "Stationery", 9, 14),

    ("Soy Wax Candle", "Home", 26, 38),
    ("Speckled Ceramic Vase", "Home", 34, 52),
    ("Linen Throw Blanket", "Home", 78, 110),
    ("Brass Picture Frame", "Home", 24, 38),
    ("Reed Diffuser", "Home", 32, 46),

    ("Gold Vermeil Hoops", "Accessories", 48, 72),
    ("Wool Felt Hat", "Accessories", 62, 88),
    ("Silk Twill Scarf", "Accessories", 54, 78),
    ("Leather Card Holder", "Accessories", 34, 48),
    ("Beaded Bracelet", "Accessories", 18, 28),

    ("Stoneware Mug", "Kitchen & Table", 16, 24),
    ("Linen Tea Towel", "Kitchen & Table", 12, 18),
    ("Olive Wood Board", "Kitchen & Table", 44, 68),
    ("Marble Salt Cellar", "Kitchen & Table", 26, 38),
    ("Enamel Stovetop Kettle", "Kitchen & Table", 68, 95),

    ("Oatmeal Soap Bar", "Bath & Body", 8, 12),
    ("Shea Hand Cream", "Bath & Body", 18, 26),
    ("Mineral Bath Salts", "Bath & Body", 22, 32),
    ("Tinted Lip Balm", "Bath & Body", 9, 14),
    ("Neroli Body Oil", "Bath & Body", 38, 56),

    ("Wildflower Honey", "Pantry & Treats", 14, 20),
    ("Sea Salt Chocolate Bar", "Pantry & Treats", 6, 10),
    ("Loose Leaf Tea Tin", "Pantry & Treats", 18, 26),
    ("Rosemary Spiced Nuts", "Pantry & Treats", 9, 14),
    ("Small-Batch Fig Jam", "Pantry & Treats", 12, 18),
]

# cost as a share of price — jewelry and bath carry the best margin, food the worst
COST_RATIO = {
    "Stationery": (0.45, 0.55),
    "Home": (0.40, 0.50),
    "Accessories": (0.32, 0.45),
    "Kitchen & Table": (0.45, 0.55),
    "Bath & Body": (0.35, 0.48),
    "Pantry & Treats": (0.55, 0.68),
}


def price_band(p):
    if p < 25:
        return "<$25"
    if p <= 75:
        return "$25-75"
    return "$75+"


def retail_price(low, high):
    """Draw in range, then snap to a retail-looking ending."""
    ending = float(np.random.choice([0.00, 0.50, 0.95], p=[0.45, 0.25, 0.30]))
    return round(np.floor(np.random.uniform(low, high)) + ending, 2)


def stock_depth(p):
    """Impulse items are stocked deep, high-ticket anchors thin."""
    if p < 25:
        return int(np.random.randint(40, 200))
    if p <= 75:
        return int(np.random.randint(12, 60))
    return int(np.random.randint(3, 20))


products = []
for i, (name, cat, lo, hi) in enumerate(CATALOG[:N_PRODUCTS], start=1):
    price = retail_price(lo, hi)
    products.append({
        "product_id": f"PROD{i:04d}",
        "product_name": name,
        "category": cat,
        "price": price,
        "price_band": price_band(price),
        "cost": round(price * np.random.uniform(*COST_RATIO[cat]), 2),
        "stock_on_hand": stock_depth(price),
    })

products_df = pd.DataFrame(products)

print(f"{len(products_df)} SKUs across {products_df['category'].nunique()} categories")
print(
    products_df.assign(margin=1 - products_df["cost"] / products_df["price"])
    .groupby("category")
    .agg(skus=("product_id", "count"), low=("price", "min"), high=("price", "max"), avg_margin=("margin", "mean"))
    .round(2)
    .to_string()
)

products_df


30 SKUs across 6 categories
                 skus    low   high  avg_margin
category                                       
Accessories         5  22.50  80.00        0.61
Bath & Body         5   9.95  50.00        0.61
Home                5  30.50  98.00        0.54
Kitchen & Table     5  13.50  93.50        0.51
Pantry & Treats     5   6.50  20.00        0.41
Stationery          5   6.95  38.95        0.51


,product_id,product_name,category,price,price_band,cost,stock_on_hand
0,PROD0001,Letterpress Greeting Card,Stationery,6.95,<$25,3.70,176
1,PROD0002,A5 Linen Notebook,Stationery,21.95,<$25,10.26,160
2,PROD0003,Brass Fountain Pen,Stationery,38.95,$25-75,18.90,37
3,PROD0004,Weekly Desk Planner,Stationery,21.00,<$25,10.99,139
4,PROD0005,Washi Tape Trio,Stationery,12.00,<$25,5.51,108
5,PROD0006,Soy Wax Candle,Home,34.00,$25-75,16.53,32
6,PROD0007,Speckled Ceramic Vase,Home,34.00,$25-75,15.78,39
7,PROD0008,Linen Throw Blanket,Home,98.00,$75+,41.72,15
8,PROD0009,Brass Picture Frame,Home,30.50,$25-75,15.21,46
9,PROD0010,Reed Diffuser,Home,41.50,$25-75,18.44,50


### Sell-through weights (best sellers)

Real boutique sales are heavily Pareto — a handful of SKUs drive most units. Picking products
uniformly would flatten that and make top-seller and 80/20 analysis meaningless, so every SKU gets
a **popularity weight** built from three things:

1. **Price elasticity** — cheap impulse items outsell expensive anchors, as `(median_price / price) ** 0.6`
2. **Hero boost** — a few deliberate best sellers get a multiplier, so the ranking isn't purely
   "cheapest wins" (the gold hoops sell well *despite* being one of the pricier SKUs)
3. **Taste jitter** — a lognormal wobble so the order isn't perfectly predictable from price

These weights drive both what gets bought (`order_items`) and what gets browsed (`product_view`
events). They are a generator input, not a column on `products` — Square's catalog wouldn't
export them.


In [43]:
# SKUs that sell above what price alone would predict
HERO_SKUS = {
    "Soy Wax Candle",
    "Letterpress Greeting Card",
    "Stoneware Mug",
    "Shea Hand Cream",
    "Gold Vermeil Hoops",
}
PRICE_ELASTICITY = 0.6  # higher = cheap items dominate more
HERO_BOOST = 3.5

_w = (products_df["price"].median() / products_df["price"]) ** PRICE_ELASTICITY
_w = _w * np.where(products_df["product_name"].isin(HERO_SKUS), HERO_BOOST, 1.0)
_w = _w * np.exp(np.random.normal(0, 0.35, len(products_df)))
product_weights = (_w / _w.sum()).to_numpy()

print("Top 8 by expected sell-through")
print(
    products_df.assign(share=product_weights)
    .nlargest(8, "share")[["product_name", "category", "price", "share"]]
    .assign(share=lambda d: (d["share"] * 100).round(1).astype(str) + "%")
    .to_string(index=False)
)


Top 8 by expected sell-through
             product_name        category  price share
Letterpress Greeting Card      Stationery   6.95 16.7%
            Stoneware Mug Kitchen & Table  21.00 10.4%
       Gold Vermeil Hoops     Accessories  48.95  8.2%
   Sea Salt Chocolate Bar Pantry & Treats   6.50  5.9%
          Tinted Lip Balm     Bath & Body   9.95  4.6%
         Oatmeal Soap Bar     Bath & Body  10.00  4.5%
          Shea Hand Cream     Bath & Body  18.95  4.5%
      Weekly Desk Planner      Stationery  21.00  4.1%


## Table 1 — `customers`  [PK: customer_id]
Only people CapFinch can actually identify (Square Customer Directory / Mailchimp audience).
`total_orders` is assigned here and counts **attributable** orders only — repeat buyers get 2.
Anonymous walk-ins never appear in this table. `signup_date` is drawn inside the history
window so every customer exists before their first order; `first_order_date` is filled in
from the actual orders once they exist.


In [44]:
def age_band(a):
    if a <= 24:
        return "18-24"
    if a <= 34:
        return "25-34"
    if a <= 44:
        return "35-44"
    return "45+"


# order-count per customer: repeat buyers get 2 orders, everyone else gets 1
order_counts = [2] * repeat_customers + [1] * (N_CUSTOMERS - repeat_customers)
random.shuffle(order_counts)

signup_window = (TODAY - pd.Timedelta(days=30) - HISTORY_START).days

customers = []
for i in range(1, N_CUSTOMERS + 1):
    age = int(np.random.randint(18, 66))
    signup = HISTORY_START + pd.Timedelta(days=int(np.random.randint(0, signup_window)))
    customers.append({
        "customer_id": f"CUST{i:04d}",
        "email": fake.unique.email(),
        "age": age,
        "age_band": age_band(age),
        "gender": random.choice(GENDERS),
        "state": fake.state_abbr(),
        "zip": fake.zipcode(),
        "signup_date": signup.date(),
        "acquisition_source": random.choice(ACQ_SOURCES),
        "first_order_date": pd.NaT,        # filled after orders are built
        "total_orders": order_counts[i - 1],
    })

customers_df = pd.DataFrame(customers)
customers_df


,customer_id,email,age,age_band,gender,state,zip,signup_date,acquisition_source,first_order_date,total_orders
0,CUST0001,johnsonjoshua@example.org,29,25-34,female,DC,97031,2025-07-05,organic,NaT,1
1,CUST0002,rhodespatricia@example.org,49,45+,female,OR,55803,2025-05-30,mailchimp,NaT,1
2,CUST0003,johnsonjeffery@example.net,30,25-34,female,PA,03979,2026-05-20,organic,NaT,1
3,CUST0004,jennifermiles@example.com,23,18-24,nonbinary,IN,59379,2025-04-11,mailchimp,NaT,1
4,CUST0005,lisa02@example.net,19,18-24,nonbinary,NE,45098,2025-04-28,referral,NaT,1
5,CUST0006,eric51@example.org,40,35-44,female,MS,13177,2026-06-09,referral,NaT,1
6,CUST0007,melanie94@example.org,48,45+,nonbinary,VA,60718,2026-07-29,social,NaT,1
7,CUST0008,dudleynicholas@example.net,36,35-44,female,LA,82898,2026-02-07,mailchimp,NaT,2
8,CUST0009,barbara10@example.net,19,18-24,nonbinary,IN,38431,2026-03-16,referral,NaT,1
9,CUST0010,wyattmichelle@example.com,28,25-34,male,KY,59930,2025-11-07,social,NaT,1


## Tables 2 & 3 — `orders` and `order_items`
One `orders` row per completed sale from **either** channel, distinguished by `channel`.
Channel-specific columns stay null on the other side:

- **online only** — `session_id`, `shipping_state`, `shipping_zip`, `shipping_fee`,
  `fulfillment_type`, `promo_code`, `device`
- **in-store only** — `register_id`, `employee_id`, `entry_method`, `tip_amount`, `receipt_type`

Money: `order_total = subtotal − discount_amount + shipping_fee`, where `subtotal` is the sum
of the order's `order_items.line_total`.

Orders are built in two passes: first the attributable ones expanded from each customer's
`total_orders`, then anonymous walk-ins and guest checkouts layered on top to hit the
identity-capture rates. Only online orders create a converting session.


In [51]:
order_rows, oi_rows, sess_rows = [], [], []
order_counter, session_counter = count(1), count(1)

product_ids = products_df["product_id"].tolist()
price_lookup = products_df.set_index("product_id")["price"].to_dict()


def build_order(customer, channel, when):
    """One order plus its line items. `customer` is None for a walk-in or guest checkout."""
    tid = f"TXN{next(order_counter):05d}"

    sizes, size_p = BASKET_SIZE[channel]
    n_items = min(int(np.random.choice(sizes, p=size_p)), len(product_ids))
    chosen = np.random.choice(product_ids, size=n_items, replace=False, p=product_weights)

    subtotal, total_qty = 0.0, 0
    for pid in chosen:
        qty = int(np.random.randint(1, 3 if channel == "online" else 4))
        unit = float(price_lookup[pid])
        line = round(qty * unit, 2)
        subtotal += line
        total_qty += qty
        oi_rows.append({
            "order_item_id": f"OI{len(oi_rows) + 1:06d}",
            "transaction_id": tid,
            "product_id": str(pid),
            "quantity": qty,
            "unit_price": unit,
            "line_total": line,
        })
    subtotal = round(subtotal, 2)

    methods, method_p = PAYMENT_METHODS[channel]
    method = str(np.random.choice(methods, p=method_p))
    declined = method != "cash" and random.random() < DECLINE_RATE[channel]  # cash never declines
    discount = round(subtotal * np.random.choice([0, 0.05, 0.10], p=[0.7, 0.2, 0.1]), 2)

    row = {
        "transaction_id": tid,
        "customer_id": customer["customer_id"] if customer is not None else None,
        "channel": channel,
        "order_datetime": when,
        "order_date": when.date(),
        "day_of_week": when.day_name(),
        "hour_of_day": when.hour,
        "subtotal": subtotal,
        "discount_amount": discount,
        "item_count": total_qty,
        "payment_method": method,
        "payment_status": "declined" if declined else "authorized",
        # online-only
        "session_id": None,
        "shipping_state": None,
        "shipping_zip": None,
        "shipping_fee": None,
        "fulfillment_type": None,
        "promo_code": None,
        "device": None,
        # in-store-only
        "register_id": None,
        "employee_id": None,
        "entry_method": None,
        "tip_amount": None,
        "receipt_type": None,
    }

    if channel == "online":
        sid = f"SESS{next(session_counter):06d}"
        fulfillment = str(np.random.choice(["ship", "pickup_in_store"], p=[0.85, 0.15]))
        free_ship = fulfillment == "pickup_in_store" or subtotal >= FREE_SHIP_THRESHOLD
        device = str(np.random.choice(DEVICES, p=DEVICE_W))
        row.update({
            "session_id": sid,
            "shipping_state": customer["state"] if customer is not None else fake.state_abbr(),
            "shipping_zip": customer["zip"] if customer is not None else fake.zipcode(),
            "shipping_fee": 0.0 if free_ship else SHIPPING_FEE,
            "fulfillment_type": fulfillment,
            "promo_code": random.choice(PROMO_CODES) if discount > 0 else None,
            "device": device,
        })
        sess_rows.append({
            "session_id": sid,
            "customer_id": row["customer_id"],
            "session_start": when - pd.Timedelta(minutes=int(np.random.randint(3, 30))),
            "device": device,
            "traffic_source": customer["acquisition_source"] if customer is not None else random.choice(ACQ_SOURCES),
            "landing_page": random.choice(LANDING_PAGES),
            "reached_cart": True,
            "converted": True,
        })
    else:
        row.update({
            "register_id": random.choice(REGISTERS),
            "employee_id": random.choice(EMPLOYEES),
            "entry_method": None if method == "cash" else str(np.random.choice(ENTRY_METHODS, p=ENTRY_METHOD_W)),
            "tip_amount": 0.0 if random.random() < 0.9 else round(subtotal * 0.05, 2),
            # a digital receipt is what links a walk-in to a customer record
            "receipt_type": random.choice(["email", "sms"]) if customer is not None else random.choice(["printed", "none"]),
        })

    row["order_total"] = round(subtotal - discount + (row["shipping_fee"] or 0.0), 2)
    return row


# Pass 1 — attributable orders. The channel split is allocated exactly rather than drawn
# per order, so the mix still holds at small N.
attributable = [cust for _, cust in customers_df.iterrows() for _ in range(int(cust["total_orders"]))]
n_attr_online = round(len(attributable) * P_ONLINE_ATTRIBUTABLE)
attr_channels = ["online"] * n_attr_online + ["in_store"] * (len(attributable) - n_attr_online)
random.shuffle(attr_channels)

for cust, channel in zip(attributable, attr_channels):
    signup = pd.Timestamp(cust["signup_date"])
    window_start = max(signup, LAUNCH_DATE) if channel == "online" else max(signup, HISTORY_START)
    order_rows.append(build_order(cust, channel, sample_datetime(channel, window_start, TODAY)))

# Pass 2 — anonymous walk-ins and guest checkouts, sized to hit the identity-capture rates
n_attr_instore = len(attributable) - n_attr_online
n_anon_instore = round(n_attr_instore * INSTORE_ANON_RATE / (1 - INSTORE_ANON_RATE))
n_guest_online = round(n_attr_online * ONLINE_GUEST_RATE / (1 - ONLINE_GUEST_RATE))

for _ in range(n_anon_instore):
    order_rows.append(build_order(None, "in_store", sample_datetime("in_store", HISTORY_START, TODAY)))
for _ in range(n_guest_online):
    order_rows.append(build_order(None, "online", sample_datetime("online", LAUNCH_DATE, TODAY)))

orders_df = pd.DataFrame(order_rows).sort_values("order_datetime").reset_index(drop=True)
order_items_df = pd.DataFrame(oi_rows)

print(f"orders={len(orders_df)}  order_items={len(order_items_df)}")
print(orders_df["channel"].value_counts().to_string())
print(f"anonymous in-store={n_anon_instore}  guest online={n_guest_online}")
print(f"online orders run {orders_df.loc[orders_df['channel'] == 'online', 'order_date'].min()} "
      f"-> {orders_df.loc[orders_df['channel'] == 'online', 'order_date'].max()}")

# rows are date-sorted, so every online order sits at the tail — preview both channels
pd.concat([
    orders_df[orders_df["channel"] == "in_store"].head(8),
    orders_df[orders_df["channel"] == "online"].head(8),
])


orders=42  order_items=91
channel
in_store    34
online       8
anonymous in-store=23  guest online=1
online orders run 2026-05-31 -> 2026-08-16


,transaction_id,customer_id,channel,order_datetime,order_date,day_of_week,hour_of_day,subtotal,discount_amount,item_count,payment_method,payment_status,session_id,shipping_state,shipping_zip,shipping_fee,fulfillment_type,promo_code,device,register_id,employee_id,entry_method,tip_amount,receipt_type,order_total
0,TXN00022,NaN,in_store,2025-04-19 13:54:21,2025-04-19,Saturday,13,10.00,1.00,1,card_present,authorized,NaN,NaN,NaN,NaN,NaN,NaN,NaN,REG01,EMP04,chip,0.0,none,9.00
1,TXN00032,NaN,in_store,2025-04-20 13:51:04,2025-04-20,Sunday,13,58.85,0.00,4,card_present,authorized,NaN,NaN,NaN,NaN,NaN,NaN,NaN,REG01,EMP03,chip,0.0,none,58.85
2,TXN00037,NaN,in_store,2025-04-25 17:19:47,2025-04-25,Friday,17,40.95,0.00,2,apple_pay,authorized,NaN,NaN,NaN,NaN,NaN,NaN,NaN,REG02,EMP02,swipe,0.0,none,40.95
3,TXN00026,NaN,in_store,2025-05-13 16:38:28,2025-05-13,Tuesday,16,97.90,9.79,2,apple_pay,authorized,NaN,NaN,NaN,NaN,NaN,NaN,NaN,REG01,EMP01,tap,0.0,none,88.11
4,TXN00028,NaN,in_store,2025-05-24 17:21:57,2025-05-24,Saturday,17,82.95,4.15,4,card_present,authorized,NaN,NaN,NaN,NaN,NaN,NaN,NaN,REG01,EMP03,chip,0.0,printed,78.80
5,TXN00020,NaN,in_store,2025-05-27 17:19:34,2025-05-27,Tuesday,17,60.00,0.00,4,card_present,authorized,NaN,NaN,NaN,NaN,NaN,NaN,NaN,REG02,EMP04,tap,0.0,printed,60.00
6,TXN00021,NaN,in_store,2025-06-30 16:32:13,2025-06-30,Monday,16,259.35,0.00,6,cash,authorized,NaN,NaN,NaN,NaN,NaN,NaN,NaN,REG02,EMP01,NaN,0.0,printed,259.35
7,TXN00012,CUST0011,in_store,2025-07-25 17:10:57,2025-07-25,Friday,17,61.00,0.00,2,card_present,authorized,NaN,NaN,NaN,NaN,NaN,NaN,NaN,REG02,EMP01,chip,0.0,email,61.00
28,TXN00008,CUST0008,online,2026-05-31 21:16:07,2026-05-31,Sunday,21,32.85,1.64,3,card_not_present,authorized,SESS000003,LA,82898,6.95,ship,MAILCHIMP15,mobile,NaN,NaN,NaN,NaN,NaN,38.16
29,TXN00018,CUST0015,online,2026-06-19 18:41:34,2026-06-19,Friday,18,43.90,0.00,2,card_not_present,authorized,SESS000007,FM,94599,6.95,ship,NaN,mobile,NaN,NaN,NaN,NaN,NaN,50.85


In [14]:
# order_items preview
order_items_df.head(15)

,order_item_id,transaction_id,product_id,quantity,unit_price,line_total
0,OI000001,TXN00001,PROD0004,2,58.18,116.36
1,OI000002,TXN00001,PROD0003,2,39.14,78.28
2,OI000003,TXN00001,PROD0005,2,57.24,114.48
3,OI000004,TXN00002,PROD0005,3,57.24,171.72
4,OI000005,TXN00003,PROD0006,3,50.66,151.98
5,OI000006,TXN00004,PROD0008,2,110.13,220.26
6,OI000007,TXN00004,PROD0006,1,50.66,50.66
7,OI000008,TXN00004,PROD0009,3,87.09,261.27
8,OI000009,TXN00005,PROD0006,2,50.66,101.32
9,OI000010,TXN00006,PROD0005,2,57.24,114.48


## Table 5 — `sessions`  [PK: session_id]
Website visits only — a POS has no equivalent, which is exactly why the funnel KPIs need this
table. The converting sessions already exist (one per **online** order). Here we add
**abandoned-cart** and **browse-only** sessions so the online funnel hits conversion ~2% and
abandonment ~68%:

- `total_carts = online_orders / (1 − 0.68)` → abandoned = carts − online orders
- `total_sessions = online_orders / 0.02` → browse-only = sessions − carts

All sessions fall on or after `LAUNCH_DATE`, and some are anonymous (`customer_id = None`).
In-store orders are deliberately excluded from both sides of the conversion ratio.


In [52]:
n_online_orders = int((orders_df["channel"] == "online").sum())
total_carts = round(n_online_orders / (1 - TARGET_ABANDONMENT))
abandoned_carts = max(total_carts - n_online_orders, 0)
total_sessions = round(n_online_orders / TARGET_CONVERSION)
browse_sessions = max(total_sessions - total_carts, 0)

cust_ids = customers_df["customer_id"].tolist()


def make_session(reached_cart, converted):
    # ~40% of non-converting traffic is a known customer, the rest is anonymous
    cust_id = random.choice(cust_ids) if random.random() < 0.4 else None
    return {
        "session_id": f"SESS{next(session_counter):06d}",
        "customer_id": cust_id,
        "session_start": sample_datetime("online", LAUNCH_DATE, TODAY),
        "device": str(np.random.choice(DEVICES, p=DEVICE_W)),
        "traffic_source": random.choice(ACQ_SOURCES),
        "landing_page": random.choice(LANDING_PAGES),
        "reached_cart": reached_cart,
        "converted": converted,
    }


for _ in range(abandoned_carts):
    sess_rows.append(make_session(reached_cart=True, converted=False))
for _ in range(browse_sessions):
    sess_rows.append(make_session(reached_cart=False, converted=False))

sessions_df = pd.DataFrame(sess_rows).sort_values("session_start").reset_index(drop=True)
print(f"sessions={len(sessions_df)}  carts={total_carts}  abandoned={abandoned_carts}  browse={browse_sessions}")
sessions_df.head(15)


sessions=400  carts=25  abandoned=17  browse=375


,session_id,customer_id,session_start,device,traffic_source,landing_page,reached_cart,converted
0,SESS000096,CUST0001,2026-05-31 13:51:24,desktop,social,/sale,False,False
1,SESS000157,CUST0002,2026-05-31 16:55:29,desktop,referral,/collections/best-sellers,False,False
2,SESS000276,NaN,2026-05-31 19:20:21,mobile,mailchimp,/new-arrivals,False,False
3,SESS000003,CUST0008,2026-05-31 20:58:07,mobile,mailchimp,/,True,True
4,SESS000061,NaN,2026-05-31 23:29:55,desktop,mailchimp,/sale,False,False
5,SESS000251,CUST0014,2026-06-01 11:25:07,desktop,referral,/sale,False,False
6,SESS000391,CUST0008,2026-06-01 19:19:23,mobile,social,/product,False,False
7,SESS000389,NaN,2026-06-02 03:57:42,mobile,mailchimp,/new-arrivals,False,False
8,SESS000134,CUST0014,2026-06-02 11:34:51,mobile,social,/new-arrivals,False,False
9,SESS000180,NaN,2026-06-02 21:09:31,mobile,mailchimp,/collections/best-sellers,False,False


### Back-fill `first_order_date`
Now that orders exist, set each customer's first order date from their earliest **attributable**
order, across both channels. This is the single source of truth for first-purchase / cohort
analysis — there is no `is_first_order` flag on `orders`, since it is derivable from here and
would be misleading anyway while most in-store sales are anonymous.


In [53]:
first_orders = orders_df.dropna(subset=["customer_id"]).groupby("customer_id")["order_datetime"].min()
customers_df["first_order_date"] = pd.to_datetime(customers_df["customer_id"].map(first_orders)).dt.date
customers_df


,customer_id,email,age,age_band,gender,state,zip,signup_date,acquisition_source,first_order_date,total_orders
0,CUST0001,johnsonjoshua@example.org,29,25-34,female,DC,97031,2025-07-05,organic,2025-12-10,1
1,CUST0002,rhodespatricia@example.org,49,45+,female,OR,55803,2025-05-30,mailchimp,2025-09-04,1
2,CUST0003,johnsonjeffery@example.net,30,25-34,female,PA,03979,2026-05-20,organic,2026-06-27,1
3,CUST0004,jennifermiles@example.com,23,18-24,nonbinary,IN,59379,2025-04-11,mailchimp,2026-08-02,1
4,CUST0005,lisa02@example.net,19,18-24,nonbinary,NE,45098,2025-04-28,referral,2026-08-16,1
5,CUST0006,eric51@example.org,40,35-44,female,MS,13177,2026-06-09,referral,2026-07-17,1
6,CUST0007,melanie94@example.org,48,45+,nonbinary,VA,60718,2026-07-29,social,2026-08-18,1
7,CUST0008,dudleynicholas@example.net,36,35-44,female,LA,82898,2026-02-07,mailchimp,2026-04-24,2
8,CUST0009,barbara10@example.net,19,18-24,nonbinary,IN,38431,2026-03-16,referral,2026-04-25,1
9,CUST0010,wyattmichelle@example.com,28,25-34,male,KY,59930,2025-11-07,social,2026-07-08,1


## Table 6 — `events`  [PK: event_id]
Funnel detail per session. Every session gets a `page_view` + some `product_view`s; sessions
that reached the cart add `add_to_cart` → `checkout_start`, and converters end with `purchase`.


In [54]:
event_rows = []


def add_event(session, etype, t, pid=None):
    event_rows.append({
        "event_id": f"EVT{len(event_rows) + 1:07d}",
        "session_id": session["session_id"],
        "event_type": etype,
        "product_id": pid,
        "event_time": t,
    })


def viewed_product():
    return str(np.random.choice(product_ids, p=product_weights))


for _, s in sessions_df.iterrows():
    t = pd.to_datetime(s["session_start"])
    add_event(s, "page_view", t)

    for _ in range(int(np.random.randint(1, 4))):
        t += pd.Timedelta(seconds=int(np.random.randint(20, 180)))
        add_event(s, "product_view", t, viewed_product())

    if s["reached_cart"]:
        t += pd.Timedelta(seconds=int(np.random.randint(20, 120)))
        add_event(s, "add_to_cart", t, viewed_product())
        t += pd.Timedelta(seconds=int(np.random.randint(20, 120)))
        add_event(s, "checkout_start", t)
        if s["converted"]:
            t += pd.Timedelta(seconds=int(np.random.randint(20, 120)))
            add_event(s, "purchase", t)

events_df = pd.DataFrame(event_rows)
print(f"events={len(events_df)}")
events_df.head(15)


events=1254


,event_id,session_id,event_type,product_id,event_time
0,EVT0000001,SESS000096,page_view,NaN,2026-05-31 13:51:24
1,EVT0000002,SESS000096,product_view,PROD0006,2026-05-31 13:53:09
2,EVT0000003,SESS000157,page_view,NaN,2026-05-31 16:55:29
3,EVT0000004,SESS000157,product_view,PROD0024,2026-05-31 16:56:26
4,EVT0000005,SESS000157,product_view,PROD0030,2026-05-31 16:59:03
5,EVT0000006,SESS000157,product_view,PROD0015,2026-05-31 17:00:25
6,EVT0000007,SESS000276,page_view,NaN,2026-05-31 19:20:21
7,EVT0000008,SESS000276,product_view,PROD0001,2026-05-31 19:21:01
8,EVT0000009,SESS000276,product_view,PROD0022,2026-05-31 19:23:27
9,EVT0000010,SESS000276,product_view,PROD0016,2026-05-31 19:25:42


## Overview, KPI check & integrity
Confirm the six DataFrames, verify the KPI targets, and assert the numeric, channel, and
timing rules hold — including that no in-store order lands outside store hours and no online
order predates launch.


In [55]:
tables = {
    "customers": customers_df,
    "orders": orders_df,
    "order_items": order_items_df,
    "products": products_df,
    "sessions": sessions_df,
    "events": events_df,
}

print("Table shapes")
for name, df in tables.items():
    print(f"  {name:12s} rows={len(df):5d}  cols={len(df.columns)}")

instore = orders_df[orders_df["channel"] == "in_store"]
online = orders_df[orders_df["channel"] == "online"]

# --- KPI check (funnel metrics are online-only) ---
conversion = len(online) / len(sessions_df)
carts = int(sessions_df["reached_cart"].sum())
abandonment = 1 - sessions_df["converted"].sum() / carts
repeat = (customers_df["total_orders"] >= 2).mean()

print("\nKPIs (actual vs target)")
print(f"  Conversion rate : {conversion:6.2%}  (target ~2%, online orders / sessions)")
print(f"  Cart abandonment: {abandonment:6.2%}  (target ~68%)")
print(f"  Repeat purchase : {repeat:6.2%}  (target ~20%)")

print("\nChannel profile")
print(f"  In-store : {len(instore):4d} orders  AOV ${instore['order_total'].mean():7.2f}  "
      f"anonymous {instore['customer_id'].isna().mean():.0%}")
print(f"  Online   : {len(online):4d} orders  AOV ${online['order_total'].mean():7.2f}  "
      f"guest {online['customer_id'].isna().mean():.0%}")

# --- Best sellers / Pareto ---
sales = (
    order_items_df.merge(products_df, on="product_id")
    .groupby(["product_name", "category"])
    .agg(units=("quantity", "sum"), revenue=("line_total", "sum"))
    .sort_values("units", ascending=False)
)
top_share = sales["revenue"].head(round(len(products_df) * 0.2)).sum() / sales["revenue"].sum()

print("\nTop 8 sellers by units")
print(sales.head(8).round(2).to_string())
print(f"\n  Top 20% of SKUs = {top_share:.0%} of revenue (Pareto check)")
print(f"  SKUs with zero sales: {len(products_df) - len(sales)}")
print("\nCategory mix by revenue")
print((sales.groupby("category")["revenue"].sum() / sales["revenue"].sum()).sort_values(ascending=False).round(3).to_string())

# --- Integrity checks ---
assert (order_items_df["line_total"] == (order_items_df["quantity"] * order_items_df["unit_price"]).round(2)).all()
recon = order_items_df.groupby("transaction_id")["line_total"].sum().round(2)
by_tid = orders_df.set_index("transaction_id").loc[recon.index]
assert np.allclose(recon.values, by_tid["subtotal"].values)
assert np.allclose(
    by_tid["order_total"].values,
    (by_tid["subtotal"] - by_tid["discount_amount"] + by_tid["shipping_fee"].fillna(0.0)).round(2).values,
)
assert orders_df["customer_id"].dropna().isin(customers_df["customer_id"]).all()
assert order_items_df["product_id"].isin(products_df["product_id"]).all()

# channel exclusivity
online_only = ["session_id", "shipping_state", "shipping_zip", "shipping_fee", "fulfillment_type", "device"]
instore_only = ["register_id", "employee_id", "tip_amount", "receipt_type"]
assert instore[online_only].isna().all().all()
assert online[instore_only].isna().all().all()
assert online["session_id"].notna().all()
assert online["session_id"].isin(sessions_df["session_id"]).all()
assert sessions_df.loc[sessions_df["session_id"].isin(online["session_id"]), "converted"].all()
assert int(sessions_df["converted"].sum()) == len(online)

# timing rules
assert instore["hour_of_day"].between(STORE_OPEN_HOUR, STORE_CLOSE_HOUR - 1).all()
assert (online["order_datetime"] >= LAUNCH_DATE).all()
assert (orders_df["order_datetime"] <= TODAY + pd.Timedelta(days=1)).all()
assert (customers_df["total_orders"] == orders_df["customer_id"].value_counts().reindex(customers_df["customer_id"]).values).all()

print("\nIntegrity checks passed: money, FKs, channel exclusivity, store hours, and launch date all consistent.")


Table shapes
  customers    rows=   15  cols=11
  orders       rows=   42  cols=25
  order_items  rows=   91  cols=6
  products     rows=   30  cols=7
  sessions     rows=  400  cols=8
  events       rows= 1254  cols=5

KPIs (actual vs target)
  Conversion rate :  2.00%  (target ~2%, online orders / sessions)
  Cart abandonment: 68.00%  (target ~68%)
  Repeat purchase : 20.00%  (target ~20%)

Channel profile
  In-store :   34 orders  AOV $ 106.36  anonymous 68%
  Online   :    8 orders  AOV $  45.56  guest 12%

Top 8 sellers by units
                                           units  revenue
product_name              category                       
Letterpress Greeting Card Stationery          21   145.95
Gold Vermeil Hoops        Accessories         13   636.35
Oatmeal Soap Bar          Bath & Body         13   130.00
Stoneware Mug             Kitchen & Table     13   273.00
Rosemary Spiced Nuts      Pantry & Treats     11   110.00
Shea Hand Cream           Bath & Body         10   189

## Export to CSV (optional)
Everything lives as DataFrames above. Uncomment to write them to a `data/` folder when ready.


In [14]:
# import os
# os.makedirs("data", exist_ok=True)
# for name, df in tables.items():
#     df.to_csv(f"data/{name}.csv", index=False)
# print("Wrote:", ", ".join(f"data/{n}.csv" for n in tables))